# Training BLIP — Image Captioning Aksara Lontara
## Model: `Salesforce/blip-image-captioning-base`

Notebook ini menjalankan **seluruh pipeline** dalam satu proses (mengikuti pola `Image_Captioning_Aksara_Lontara.ipynb`):
1. Mount Google Drive & Install Library
2. Pipeline: Persiapan Dataset → Training → Evaluasi → Simpan ke Drive

**Konfigurasi:** Batch size 8 · Epochs 20 · LR 2e-5 · AdamW · MAX_LENGTH 128

> **Pastikan runtime menggunakan GPU** (Runtime → Change runtime type → GPU).
> Semua `import` berada di dalam sel Pipeline agar sel dapat dijalankan mandiri (tahan terhadap reset runtime).

---
## 1. Setup: Mount Google Drive & Install Library

In [ ]:
# ═══════════════════════════════════════════════════════════════
# SETUP: Mount Google Drive & Install Library
# ═══════════════════════════════════════════════════════════════
import os
try:
    os.chdir("/content")
except OSError:
    pass

from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/ImageCaptioning"
LOCAL_DIR = "/content/ImageCaptioning"

# Copy dataset dari Drive ke local (lebih stabil & cepat)
assert os.path.exists(DRIVE_DIR), f"ERROR: {DRIVE_DIR} tidak ditemukan!"
!rm -rf {LOCAL_DIR}
!cp -r {DRIVE_DIR} {LOCAL_DIR}
os.chdir(LOCAL_DIR)
print(f"✅ Working directory: {os.getcwd()}")
print(f"✅ Isi folder: {os.listdir('.')}")

# Install library
!pip install -q transformers Pillow nltk rouge-score tqdm

import torch
print(f"✅ PyTorch: {torch.__version__}")
print(f"✅ CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# PIPELINE LENGKAP: Prepare → Train → Evaluate → Save (satu proses)
# Semua import berada di sel ini agar dapat dijalankan mandiri
# (tahan terhadap reset runtime Colab).
# ═══════════════════════════════════════════════════════════════
import os
import json
import random
import shutil
import warnings

import torch
import numpy as np
import matplotlib
matplotlib.rcParams["font.size"] = 11
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    BlipProcessor,
    BlipForConditionalGeneration,
    get_linear_schedule_with_warmup,
)
from tqdm import tqdm
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score as calc_meteor
from rouge_score import rouge_scorer
import nltk

warnings.filterwarnings("ignore")
nltk.download("wordnet", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("omw-1.4", quiet=True)

# ─── Konfigurasi ───
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

RANDOM_SEED = 42
IMAGE_SIZE = 384
MAX_LENGTH = 128
NUM_EPOCHS = 20
LR = 2e-5
BATCH_SIZE = 8
MODEL_NAME = "Salesforce/blip-image-captioning-base"
MODEL_DIR = "model/blip"
RAW_IMAGE_DIR = "dataset/images"
CAPTION_FILE = "dataset/captions.json"
OUTPUT_DIR = "dataset/processed"
EVAL_DIR = "hasil_evaluasi"

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(RANDOM_SEED)

os.makedirs(EVAL_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# Mapping ltr-XX -> {karakter, vokal}
KARAKTER_DASAR = [
    "a", "ba", "ca", "da", "ga",
    "ha", "ja", "ka", "la", "ma",
    "na", "nga", "nya", "pa", "ra",
    "sa", "ta", "wa", "ya"
]
VOKAL = ["a", "i", "u", "e", "o"]
mapping = {}
_idx = 1
for _dasar in KARAKTER_DASAR:
    for _vokal in VOKAL:
        mapping[f"ltr-{_idx:02d}"] = {"karakter": _dasar, "vokal": _vokal}
        _idx += 1


# ═══════════════════════════════════════════════════════════════
# TAHAP 1: PERSIAPAN DATASET
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("  TAHAP 1: PERSIAPAN DATASET")
print("=" * 60)

with open(CAPTION_FILE, "r", encoding="utf-8") as f:
    raw_data = json.load(f)
print(f"Entri caption: {len(raw_data)}")

flat_data = []
for item in raw_data:
    img_id = item["images"]
    img_filename = f"{img_id}.png"
    info = mapping.get(img_id, {})
    for caption in item["captions"]:
        flat_data.append({
            "image": img_filename,
            "caption": caption,
            "karakter": info.get("karakter", ""),
            "vokal": info.get("vokal", ""),
        })
print(f"Total pasang (image, caption): {len(flat_data)}")

# Split per gambar (hindari kebocoran antar split)
unique_images = []
for item in flat_data:
    if item["image"] not in unique_images:
        unique_images.append(item["image"])
random.shuffle(unique_images)
n = len(unique_images)
n_train = int(n * 0.8)
n_val = int(n * 0.1)
train_imgs = set(unique_images[:n_train])
val_imgs = set(unique_images[n_train:n_train + n_val])
test_imgs = set(unique_images[n_train + n_val:])

train_data = [d for d in flat_data if d["image"] in train_imgs]
val_data = [d for d in flat_data if d["image"] in val_imgs]
test_data = [d for d in flat_data if d["image"] in test_imgs]
print(f"Split: Train={len(train_data)}, Val={len(val_data)}, Test={len(test_data)}")

for name, data in [("train", train_data), ("val", val_data), ("test", test_data)]:
    os.makedirs(f"{OUTPUT_DIR}/{name}/images", exist_ok=True)
    with open(f"{OUTPUT_DIR}/{name}/captions_{name}.json", "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

for name, data in [("train", train_data), ("val", val_data), ("test", test_data)]:
    processed = set()
    errors = []
    for item in data:
        img = item["image"]
        if img in processed:
            continue
        processed.add(img)
        src = os.path.join(RAW_IMAGE_DIR, img)
        dst = os.path.join(OUTPUT_DIR, name, "images", img)
        if os.path.exists(src):
            try:
                im = Image.open(src).convert("RGB")
                im = im.resize((IMAGE_SIZE, IMAGE_SIZE), Image.LANCZOS)
                im.save(dst, format="PNG")
                Image.open(dst).verify()
            except Exception as e:
                errors.append(f"{img}: {e}")
        else:
            errors.append(f"{img}: file tidak ditemukan")
    print(f"  {name}: {len(processed)} gambar" + (f" ({len(errors)} error)" if errors else ""))

print("✅ Persiapan dataset selesai!")


# ═══════════════════════════════════════════════════════════════
# TAHAP 2: FINE-TUNING BLIP
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("  TAHAP 2: FINE-TUNING BLIP")
print("=" * 60)


class CaptionDataset(Dataset):
    """Dataset untuk BLIP."""
    def __init__(self, json_file, image_dir, processor, max_length=128):
        with open(json_file, "r", encoding="utf-8") as f:
            self.data = json.load(f)
        self.image_dir = image_dir
        self.processor = processor
        self.max_length = max_length
        print(f"  Dataset: {len(self.data)} pasang")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        img_path = os.path.join(self.image_dir, os.path.basename(item["image"]))
        try:
            image = Image.open(img_path).convert("RGB")
        except Exception:
            image = Image.new("RGB", (IMAGE_SIZE, IMAGE_SIZE), (255, 255, 255))
        encoding = self.processor(
            images=image, text=item["caption"],
            padding="max_length", truncation=True,
            max_length=self.max_length, return_tensors="pt"
        )
        input_ids = encoding["input_ids"].squeeze()
        pixel_values = encoding["pixel_values"].squeeze()
        labels = input_ids.clone()
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        result = {"pixel_values": pixel_values, "input_ids": input_ids, "labels": labels}
        if "attention_mask" in encoding:
            result["attention_mask"] = encoding["attention_mask"].squeeze()
        return result


# Load model
print("Memuat BLIP...")
processor = BlipProcessor.from_pretrained(MODEL_NAME)
model = BlipForConditionalGeneration.from_pretrained(MODEL_NAME).to(DEVICE)
print(f"Parameter: {sum(p.numel() for p in model.parameters()):,}")

# Simpan processor
processor.save_pretrained(f"{MODEL_DIR}/best_model")

# DataLoader
train_ds = CaptionDataset(f"{OUTPUT_DIR}/train/captions_train.json", f"{OUTPUT_DIR}/train/images", processor, MAX_LENGTH)
val_ds = CaptionDataset(f"{OUTPUT_DIR}/val/captions_val.json", f"{OUTPUT_DIR}/val/images", processor, MAX_LENGTH)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

# Training
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * NUM_EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=100, num_training_steps=total_steps)

history = {"train_loss": [], "val_loss": []}
best_val_loss = float("inf")

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} Train", leave=False):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    train_loss = total_loss / len(train_loader)

    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} Val", leave=False):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            outputs = model(**batch)
            total_loss += outputs.loss.item()
    val_loss = total_loss / len(val_loader)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        os.makedirs(f"{MODEL_DIR}/best_model", exist_ok=True)
        model.save_pretrained(f"{MODEL_DIR}/best_model")
        print(f"  -> Best model! Val Loss: {best_val_loss:.4f}")

    if epoch % 5 == 0:
        ckpt = f"{MODEL_DIR}/checkpoint_epoch_{epoch}"
        os.makedirs(ckpt, exist_ok=True)
        model.save_pretrained(ckpt)

with open(f"{MODEL_DIR}/training_history.json", "w") as f:
    json.dump(history, f, indent=2)

print(f"\nTraining selesai! Best Val Loss: {best_val_loss:.4f}")


# ═══════════════════════════════════════════════════════════════
# TAHAP 3: EVALUASI MODEL (TEST SET)
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("  TAHAP 3: EVALUASI MODEL (TEST SET)")
print("=" * 60)

# Plot kurva training (Train vs Val Loss)
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
epochs_range = range(1, len(history["train_loss"]) + 1)
ax.plot(epochs_range, history["train_loss"], "o-", label="Train Loss", color="#2196F3")
ax.plot(epochs_range, history["val_loss"], "s-", label="Val Loss", color="#FF5722")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("BLIP - Training & Validation Loss", fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{EVAL_DIR}/training_curve_blip.png", dpi=150, bbox_inches="tight")
plt.show()

# Muat best model untuk evaluasi
best_dir = f"{MODEL_DIR}/best_model"
eval_processor = BlipProcessor.from_pretrained(best_dir)
eval_model = BlipForConditionalGeneration.from_pretrained(best_dir).to(DEVICE)
eval_model.eval()
print("Model evaluasi dimuat (best model).")

with open(f"{OUTPUT_DIR}/test/captions_test.json", "r", encoding="utf-8") as f:
    test_items = json.load(f)
print(f"Data test: {len(test_items)} entri")

# Kelompokkan per gambar + ambil karakter/vokal dari mapping
gambar_dict = {}
for item in test_items:
    key = item["image"]
    if key not in gambar_dict:
        ltr_id = os.path.splitext(os.path.basename(key))[0]
        info = mapping.get(ltr_id, {})
        gambar_dict[key] = {"captions": [], "karakter": info.get("karakter", ""), "vokal": info.get("vokal", "")}
    gambar_dict[key]["captions"].append(item["caption"])

# Generate & kumpulkan referensi/hipotesis
referensi_list = []
hipotesis_list = []
hasil_detail = []
for image_key, info in tqdm(gambar_dict.items(), desc="Evaluasi Test Set"):
    img_path = os.path.join(f"{OUTPUT_DIR}/test/images", os.path.basename(image_key))
    try:
        image = Image.open(img_path).convert("RGB")
    except Exception:
        continue
    inputs = eval_processor(images=image, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        output = eval_model.generate(**inputs, max_length=MAX_LENGTH, num_beams=4, early_stopping=True)
    pred = eval_processor.decode(output[0], skip_special_tokens=True)
    refs = [cap.lower().split() for cap in info["captions"]]
    hyp = pred.lower().split()
    referensi_list.append(refs)
    hipotesis_list.append(hyp)
    hasil_detail.append({
        "image": image_key, "karakter": info["karakter"], "vokal": info["vokal"],
        "caption_prediksi": pred, "captions_referensi": info["captions"]
    })

# Hitung metrik (BLEU-1..4, METEOR, ROUGE-L)
smoother = SmoothingFunction().method1
bleu1 = corpus_bleu(referensi_list, hipotesis_list, weights=(1, 0, 0, 0), smoothing_function=smoother)
bleu2 = corpus_bleu(referensi_list, hipotesis_list, weights=(0.5, 0.5, 0, 0), smoothing_function=smoother)
bleu3 = corpus_bleu(referensi_list, hipotesis_list, weights=(0.33, 0.33, 0.33, 0), smoothing_function=smoother)
bleu4 = corpus_bleu(referensi_list, hipotesis_list, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoother)
meteor_avg = np.mean([calc_meteor(refs, hyp) for refs, hyp in zip(referensi_list, hipotesis_list)])
rouge_sc = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
rouge_scores_list = []
for refs, hyp in zip(referensi_list, hipotesis_list):
    hyp_str = " ".join(hyp)
    rouge_scores_list.append(max(rouge_sc.score(" ".join(r), hyp_str)["rougeL"].fmeasure for r in refs))
rouge_avg = np.mean(rouge_scores_list)

metrik = {
    "BLEU-1": bleu1 * 100, "BLEU-2": bleu2 * 100,
    "BLEU-3": bleu3 * 100, "BLEU-4": bleu4 * 100,
    "METEOR": meteor_avg * 100, "ROUGE-L": rouge_avg * 100
}

print("\n" + "=" * 60)
print("  HASIL EVALUASI — TEST SET")
print("=" * 60)
print("┌────────────┬──────────┐")
print("│   Metrik   │   Skor   │")
print("├────────────┼──────────┤")
for k, v in metrik.items():
    print(f"│ {k:<10} │  {v:>5.2f}%  │")
print("└────────────┴──────────┘")

# Akurasi identifikasi per karakter dasar
print("\n  Akurasi Identifikasi per Karakter Dasar:")
karakter_stats = {}
for item in hasil_detail:
    kar = item["karakter"]
    if not kar:
        continue
    karakter_stats.setdefault(kar, {"total": 0, "benar": 0})
    karakter_stats[kar]["total"] += 1
    if kar.lower() in item["caption_prediksi"].lower():
        karakter_stats[kar]["benar"] += 1

print("┌──────────────┬───────┬────────┬──────────┐")
print("│   Karakter   │ Total │ Benar  │ Akurasi  │")
print("├──────────────┼───────┼────────┼──────────┤")
total_benar = 0
total_semua = 0
for kar in sorted(karakter_stats.keys()):
    s = karakter_stats[kar]
    akurasi = (s["benar"] / s["total"] * 100) if s["total"] > 0 else 0
    total_benar += s["benar"]
    total_semua += s["total"]
    print(f"│ {kar:<12} │  {s['total']:>3}  │   {s['benar']:>3}  │  {akurasi:>5.1f}%  │")
print("├──────────────┼───────┼────────┼──────────┤")
akurasi_total = (total_benar / total_semua * 100) if total_semua > 0 else 0
print(f"│ {'TOTAL':<12} │  {total_semua:>3}  │   {total_benar:>3}  │  {akurasi_total:>5.1f}%  │")
print("└──────────────┴───────┴────────┴──────────┘")

# Visualisasi metrik + akurasi per karakter
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
metrik_names = list(metrik.keys())
metrik_vals = list(metrik.values())
bars = ax1.bar(metrik_names, metrik_vals,
               color=["#2196F3", "#2196F3", "#2196F3", "#2196F3", "#4CAF50", "#FF9800"])
ax1.set_ylabel("Score (%)")
ax1.set_title("Evaluation Metrics — Test Set")
ax1.set_ylim(0, 100)
ax1.grid(True, alpha=0.3, axis="y")
for bar, val in zip(bars, metrik_vals):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
             f"{val:.1f}%", ha="center", va="bottom", fontsize=9, fontweight="bold")
if karakter_stats:
    kar_names = sorted(karakter_stats.keys())
    kar_akurasi = [(karakter_stats[k]["benar"] / karakter_stats[k]["total"] * 100)
                   if karakter_stats[k]["total"] > 0 else 0 for k in kar_names]
    bar_colors = ["#4CAF50" if a >= 80 else "#FF9800" if a >= 50 else "#F44336" for a in kar_akurasi]
    ax2.bar(kar_names, kar_akurasi, color=bar_colors)
    ax2.set_ylabel("Akurasi (%)")
    ax2.set_title("Akurasi Identifikasi per Karakter Dasar")
    ax2.set_ylim(0, 105)
    ax2.axhline(y=80, color="green", linestyle="--", alpha=0.5, label="Target 80%")
    ax2.legend()
    ax2.grid(True, alpha=0.3, axis="y")
    plt.setp(ax2.get_xticklabels(), rotation=45, ha="right")
plt.tight_layout()
plt.savefig(f"{EVAL_DIR}/evaluasi_metrik_blip.png", dpi=150, bbox_inches="tight")
plt.show()

# Simpan hasil evaluasi lengkap
hasil_final = {
    "metrik": {k: round(v, 2) for k, v in metrik.items()},
    "akurasi_per_karakter": {k: round((v["benar"] / v["total"] * 100) if v["total"] > 0 else 0, 1)
                             for k, v in karakter_stats.items()},
    "training_history": {
        "best_val_loss": round(best_val_loss, 4),
        "train_loss": history["train_loss"],
        "val_loss": history["val_loss"]
    },
    "detail": hasil_detail
}
with open(f"{EVAL_DIR}/evaluasi_blip.json", "w", encoding="utf-8") as f:
    json.dump(hasil_final, f, ensure_ascii=False, indent=2)
print(f"\nHasil evaluasi disimpan: {EVAL_DIR}/evaluasi_blip.json")


# ═══════════════════════════════════════════════════════════════
# TAHAP 4: INFERENCE & VISUALISASI (contoh gambar test)
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("  TAHAP 4: INFERENCE & VISUALISASI")
print("=" * 60)
test_folder = f"{OUTPUT_DIR}/test/images"
test_images = sorted([f for f in os.listdir(test_folder) if f.endswith(".png")])[:6]
inference_results = []
n_show = len(test_images)
if n_show > 0:
    rows = 2 if n_show > 3 else 1
    cols = int(np.ceil(n_show / rows))
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
    axes = np.array(axes).flatten()
    for idx, img_name in enumerate(test_images):
        img_path = os.path.join(test_folder, img_name)
        image = Image.open(img_path).convert("RGB")
        inputs = eval_processor(images=image, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            output = eval_model.generate(**inputs, max_length=MAX_LENGTH, num_beams=4,
                                         early_stopping=True, no_repeat_ngram_size=2)
        caption = eval_processor.decode(output[0], skip_special_tokens=True)
        inference_results.append({"gambar": img_name, "caption": caption})
        print(f"  {img_name} → {caption}")
        axes[idx].imshow(image)
        axes[idx].set_title(caption, fontsize=9, wrap=True)
        axes[idx].axis("off")
    for idx in range(n_show, len(axes)):
        axes[idx].axis("off")
    plt.tight_layout()
    plt.savefig(f"{EVAL_DIR}/visualisasi_prediksi_blip.png", dpi=150, bbox_inches="tight")
    plt.show()
with open(f"{EVAL_DIR}/hasil_inference_blip.json", "w", encoding="utf-8") as f:
    json.dump(inference_results, f, ensure_ascii=False, indent=2)


# ═══════════════════════════════════════════════════════════════
# TAHAP 5: SIMPAN KE GOOGLE DRIVE
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("  TAHAP 5: SIMPAN KE GOOGLE DRIVE")
print("=" * 60)
DRIVE_DIR = "/content/drive/MyDrive/ImageCaptioning"
LOCAL_DIR = "/content/ImageCaptioning"
for folder in [MODEL_DIR, EVAL_DIR]:
    src = os.path.join(LOCAL_DIR, folder)
    dst = os.path.join(DRIVE_DIR, folder)
    if os.path.exists(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f"  {folder}/ → Drive")
print("\n✅ SELURUH PIPELINE SELESAI! Semua hasil tersimpan di Google Drive.")